# Time-boxed DPO research run (Colab T4)

Steps 1 and 2 are already complete. Before running: commit and push the intended repository changes, add a Colab Secret named `HF_TOKEN` with write access, and edit `HUB_REPO_ID` below to a private Hub model repository you own. This notebook does not use Google Drive. Checkpoints and logs are periodically uploaded to the Hub; results and per-step training CSVs are written locally under `results/scaled-t4-512` and snapshots are uploaded. Do not print or save the token.

The configured plan uses 512 training pairs, 128-row margin/random subsets, one seed (42), batch 1, accumulation 8, 512-token sequences, and six validation checkpoints with generation capped at 256 tokens. It uses beta 0.1 for the selected/random arms and shares the full beta-0.1 arm as the full-data comparator. UltraFeedback is pinned to commit `3949bf5f8c17c394422ccfab0c31ea9c20bdeb85`. The 50-step T4 smoke measured about 3.06 s/SFT step and 6.12 s/DPO step. One seed is used because validation generation and Hub transfer add substantial unmeasured time. This is an estimate, not a guarantee of the 3-hour limit.

Re-run the final cell after a disconnect. The runner restores Hub snapshots, skips completed stages, and resumes incomplete training from the newest uploaded optimizer checkpoint. It needs the same config and Hub repository.

In [ ]:
from pathlib import Path
import os, subprocess, sys
from google.colab import userdata
REPO_URL = 'https://github.com/Adhilmohdkt/dpo-from-scratch.git'
BRANCH = 'main'  # Use the branch containing the reviewed changes.
PROJECT_DIR = Path('/content/dpo-from-scratch')
if PROJECT_DIR.exists():
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only', 'origin', BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(PROJECT_DIR)], check=True)
try:
    token = userdata.get('HF_TOKEN')
except Exception as exc:
    raise RuntimeError('Add a write-enabled HF_TOKEN in Colab Secrets, then rerun this cell.') from exc
if not token:
    raise RuntimeError('Add a write-enabled HF_TOKEN in Colab Secrets, then rerun this cell.')
os.environ['HF_TOKEN'] = token
del token
%cd /content/dpo-from-scratch

In [ ]:
import subprocess, sys, torch, shutil
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print({'python': sys.version.split()[0], 'torch': torch.__version__, 'cuda_available': torch.cuda.is_available(), 'gpu': GPU_NAME})
if not torch.cuda.is_available() or 'T4' not in GPU_NAME:
    raise RuntimeError('This run is configured for an active Tesla T4. Select a T4 accelerator and restart.')
managed_torch = torch.__version__
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True)
from importlib.metadata import version
if version('torch').split('+')[0] != managed_torch.split('+')[0]:
    raise RuntimeError('Dependency installation changed Colab-managed PyTorch; stop and reconnect to a fresh runtime.')
import transformers, peft, datasets, trl, accelerate, yaml
print({'transformers': transformers.__version__, 'peft': peft.__version__, 'datasets': datasets.__version__, 'trl': trl.__version__, 'accelerate': accelerate.__version__})

In [ ]:
import yaml
CONFIG_PATH = Path('configs/scaled_t4.yaml')
HUB_REPO_ID = 'YOUR_HF_USERNAME/dpo-research-artifacts'  # Private repository owned by you.
if HUB_REPO_ID.startswith('YOUR_'):
    raise ValueError('Edit HUB_REPO_ID to your private Hugging Face repository before running.')
config = yaml.safe_load(CONFIG_PATH.read_text(encoding='utf-8'))
config['hub']['repo_id'] = HUB_REPO_ID
CONFIG_PATH.write_text(yaml.safe_dump(config, sort_keys=False), encoding='utf-8')
print('Configured run:', {'config': str(CONFIG_PATH), 'training_pairs': config['data']['max_train_samples'], 'subset_size': config['experiment']['selection_subset_size'], 'seeds': config['experiment']['seeds'], 'hub_repo': HUB_REPO_ID})

In [ ]:
# Start or resume the scaled experiment. The runner validates HF_TOKEN without displaying its value.
subprocess.run([sys.executable, '-m', 'scripts.run_scaled_experiments', '--config', str(CONFIG_PATH)], check=True)
print('Run outputs:', Path(config['experiment']['output_dir']).resolve())